# Phone photos to a labeled steel-defect dataset
Connect the Google account that owns **My Drive → ARM Steel External Dataset** when Colab prompts you to mount Drive. The notebook reads the ten JPEGs and **archive (1).zip** directly from that folder. Use a Colab CPU runtime. Run in order and stop at the annotation step to enter your own boxes.

The working format is **200 × 200 grayscale PNG**. The existing Jetson pipeline prepares the separate **416 × 416** model tensor. Crops match the reference geometry without stretching. They cannot recover detail lost to blur or glare, and these phone photos are from a different imaging setup.

In [ ]:
#@title Load the supplied photos and upload the NEU-DET ZIP
from google.colab import drive
from pathlib import Path
from io import BytesIO
from tempfile import TemporaryDirectory
import csv, json, math, shutil, zipfile
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps

TARGET = 200
CLASSES = ('crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches')
def decode(data):
    with Image.open(BytesIO(data)) as source:
        return ImageOps.exif_transpose(source).copy()

def photo_id(name):
    return Path(name).stem.split(' at ')[-1]

def write_csv(path, columns, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.writer(handle)
        writer.writerow(columns)
        writer.writerows(rows)

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/ARM Steel External Dataset')
REFERENCE_ZIP = DATA_DIR / 'archive (1).zip'
photo_paths = sorted(DATA_DIR.glob('WhatsApp Image*.jpeg'))
if not REFERENCE_ZIP.is_file() or not photo_paths:
    raise FileNotFoundError(f"Expected archive (1).zip and the original JPEGs in {DATA_DIR}")
photos = {photo_id(path.name): decode(path.read_bytes()) for path in photo_paths}
source_names = {photo_id(path.name): path.name for path in photo_paths}

reference_zip = zipfile.ZipFile(REFERENCE_ZIP)
ref_paths = {Path(name).stem: name for name in reference_zip.namelist()
             if Path(name).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}}
xml_paths = {Path(name).stem: name for name in reference_zip.namelist()
             if name.lower().endswith('.xml')}
reference_ids = sorted(ref_paths.keys() & xml_paths.keys())
if not reference_ids:
    raise ValueError("The archive must contain NEU-DET image/XML pairs with matching stems.")
print(f"{len(photos)} phone photos and {len(reference_ids)} reference image/XML pairs")

## Choose a square region on the original
Starter regions below were picked visually from your photos. Review the grid before using them. Coordinates are in the original photo: `(left, top, side)`. Multiple crops from the same photo can be added by assigning another crop ID. A region picked because a defect is visible is a **selected-crop demo**. For a wider field-of-view test, define a fixed steel region or tile grid before inference and keep misses.

Start with **07b** for the long diagonal scratch and **07a** for the sheet with several visible lines. **02b/03a**, **00a/02a**, and **05a/08a/08b** appear to contain repeated views. Preserve these provisional groups and check which physical pieces actually repeat. Do not count near-identical crops as independent evidence.

In [ ]:
# crop ID: (photo suffix, left, top, square side, provisional source group)
CROPS = {
    '00a': ('09.34.00', 35, 65, 365, 'cutout'),
    '02a': ('09.34.02 (1)', 35, 65, 365, 'cutout'),
    '02b': ('09.34.02', 65, 500, 385, 'reflective_sheet'),
    '03a': ('09.34.03', 65, 500, 385, 'reflective_sheet'),
    '05a': ('09.34.05', 97, 20, 248, 'machined_parts'),
    '07a': ('09.34.07', 132, 586, 435, 'scratched_sheet'),
    '07b': ('09.34.07 (1)', 50, 476, 455, 'diagonal_scratch'),
    '08a': ('09.34.08', 190, 541, 230, 'machined_parts'),
    '08b': ('09.34.08 (1)', 80, 22, 240, 'machined_parts'),
    '09a': ('09.34.09', 47, 481, 247, 'cut_plate'),
}
USE_AUTOCONTRAST = False  # Keep off for the baseline. Freeze this before annotation/evaluation.

def grid(axis, image, title, step):
    width, height = image.size
    axis.imshow(image, cmap='gray', vmin=0, vmax=255,
                extent=(0, width, height, 0))
    axis.set(xticks=range(0, width+1, step), yticks=range(0, height+1, step), title=title)
    axis.grid(alpha=.3)

crops = {}
for key, (photo, x, y, side, group) in CROPS.items():
    if not key.isalnum():
        raise ValueError(f"{key}: use an alphanumeric crop ID.")
    original = photos[photo]
    if not (side > 0 and x >= 0 and y >= 0 and x+side <= original.width and y+side <= original.height):
        raise ValueError(f"{key}: crop is outside the photo.")
    gray = original.crop((x,y,x+side,y+side)).convert('L')
    resized = gray.resize((TARGET,TARGET), Image.Resampling.LANCZOS)
    crops[key] = ImageOps.autocontrast(resized, cutoff=1) if USE_AUTOCONTRAST else resized

fig, axes = plt.subplots(len(crops), 2, figsize=(10, 3.4*len(crops)), squeeze=False)
for (key, image), (original_axis, crop_axis) in zip(crops.items(), axes):
    photo, x, y, side, group = CROPS[key]
    grid(original_axis, photos[photo], photo, 200)
    original_axis.add_patch(Rectangle((x,y), side,side, fill=False, edgecolor='lime', linewidth=2))
    grid(crop_axis, image, key, 25)
fig.tight_layout()
plt.show()

## Compare against your NEU-DET archive
Inspect actual labeled examples from all six classes. The printed percentiles compare brightness and contrast, and the near-white fraction helps flag possible glare. These are diagnostics. Similar numbers alone do not establish equivalent image quality. Look at scratch width, background texture, shadows and glare as well.

In [ ]:
def ref_image(key):
    return decode(reference_zip.read(ref_paths[key])).convert('L')

def reference_boxes(key):
    root = ET.fromstring(reference_zip.read(xml_paths[key]))
    return [(obj.findtext('name'), tuple(float(obj.findtext('bndbox/'+field))
            for field in ('xmin','ymin','xmax','ymax'))) for obj in root.findall('object')]

def draw_box(axis, label, box):
    x1,y1,x2,y2 = box
    axis.add_patch(Rectangle((x1,y1), x2-x1,y2-y1, fill=False, edgecolor='lime', linewidth=1.5))
    axis.text(x1,y1,label,color='white',fontsize=7,backgroundcolor='black',va='top')

rng = np.random.default_rng(42)
show_refs = {}
study = []
for label in CLASSES:
    matches = [key for key in reference_ids if key.startswith(label+'_')]
    if not matches:
        raise ValueError(f"No reference filename matches {label}.")
    selected = rng.choice(matches, size=min(30,len(matches)), replace=False)
    show_refs[label] = selected[0]
    study.extend(ref_image(key) for key in selected)

def image_stats(image):
    arr = np.asarray(image)
    p05, p50, p95 = np.percentile(arr, [5,50,95])
    return np.array([p05,p50,p95,p95-p05,100*np.mean(arr>=250)])

print('Median per-image p05 / p50 / p95 / contrast span / % near-white')
print(f'NEU-DET ({len(study)} images):', np.median([image_stats(im) for im in study],axis=0).round(2))
print(f'Phone crops ({len(crops)} images):', np.median([image_stats(im) for im in crops.values()],axis=0).round(2))
print('Inspection of individual crops:')
for key,image in crops.items():
    print(key, image_stats(image).round(2))

fig, axes = plt.subplots(2,3,figsize=(12,8))
for axis, (label,key) in zip(axes.flat,show_refs.items()):
    grid(axis, ref_image(key), key, 50)
    for name,box in reference_boxes(key):
        draw_box(axis,name,box)
fig.tight_layout()
plt.show()

keys = [key for key in ('07b','07a','03a') if key in crops]
if keys:
    fig, axes = plt.subplots(1,len(keys)+1, figsize=(4*(len(keys)+1),4),squeeze=False)
    grid(axes[0,0], ref_image(show_refs['scratches']), 'NEU-DET scratch', 25)
    for axis,key in zip(axes[0,1:],keys):
        grid(axis,crops[key],key,25)
    fig.tight_layout()
    plt.show()

## Check the entire defect on the original photo
Use the original-photo grid to trace the defect from end to end. In `REVIEW`, enter boxes as `(class_name, left, top, right, bottom)` in **original photo pixels**. For a diagonal scratch, include both tips and its full width inside one tight axis-aligned box. Follow the visible defect, including any faint continuation. If the photo itself cuts off the defect or glare hides its end, exclude that crop or retake the photo.

The preview shows the original and its crop. A positive box must sit at least `CROP_MARGIN` original pixels inside the crop on all sides. If it fails, enlarge or shift that crop in `CROPS`, then rerun the crop cell and review. This checks the box you drew. Human review still determines whether the box contains the whole visible defect.

Use **positive** for reviewed crops with all supported defects labeled, **negative** for reviewed crops with none, and **exclude** for uncertain examples. Crops absent from `REVIEW` are unreviewed. Deliberate holes, grooves and reflections do not establish a trained defect class.

In [ ]:
SHOW = '07b'
CROP_MARGIN = 5
REVIEW = {


}

def checked(key, *, enforce_margin=True):
    status, objects = REVIEW.get(key, ('unreviewed', []))
    if status not in {'positive','negative','exclude','unreviewed'} or bool(objects) != (status=='positive'):
        raise ValueError(f"{key}: positive crops need boxes; other statuses use an empty list.")
    photo, left, top, side, _ = CROPS[key]
    original = photos[photo]
    converted = []
    for label, x1, y1, x2, y2 in objects:
        if label not in CLASSES or not (0 <= x1 < x2 <= original.width and 0 <= y1 < y2 <= original.height):
            raise ValueError(f"{key}: invalid class or original-photo coordinates.")
        if not (left+CROP_MARGIN <= x1 < x2 <= left+side-CROP_MARGIN
                and top+CROP_MARGIN <= y1 < y2 <= top+side-CROP_MARGIN):
            message = f"{key}: {label} box is clipped or too close to the crop edge. Adjust CROPS and rerun."
            if enforce_margin:
                raise ValueError(message)
            print(message)
        scale = TARGET/side
        converted.append((label, (x1-left)*scale, (y1-top)*scale,
                          (x2-left)*scale, (y2-top)*scale))
    return status, converted

photo, left, top, side, _ = CROPS[SHOW]
status, converted = checked(SHOW, enforce_margin=False)
fig, (original_axis, crop_axis) = plt.subplots(1, 2, figsize=(13, 10))
grid(original_axis, photos[photo], f"{photo} | original pixels", 100)
original_axis.add_patch(Rectangle((left,top), side,side, fill=False, edgecolor='cyan', linewidth=2))
for label,*box in REVIEW.get(SHOW, ('unreviewed', []))[1]:
    draw_box(original_axis,label,box)
grid(crop_axis, crops[SHOW], f"{SHOW} | {status}", 20)
for label,*box in converted:
    draw_box(crop_axis,label,box)
fig.tight_layout()
plt.show()

## Export for Jetson and evaluation
For a quick demo, export with `CLASS_ORDER = []`. This still saves reviewed boxes in CSV, along with all demo images. To generate YOLO .txt labels, paste the six class names **in increasing model ID order** from the training YAML. This notebook uses NEU XML spellings, so normalize spaces/underscores and singular/plural names to the spellings above while keeping their order. XML names alone do not establish ID order.

Copy **demo/images/** to the Jetson for notebook 05, or use **evaluation/images/** for the reviewed batch. Keep **evaluation/labels/** paired with those images for scoring. YOLO uses normalized center-x, center-y, width and height. A reviewed negative gets an empty label file. Compare predictions to ground truth with one-to-one class-matched boxes (for example IoU ≥ 0.50), and report precision/recall plus the positive, negative and source-group counts. A selected clip is qualitative evidence.

**annotated/images/** contains PNG copies with visible boxes for reviewed positives. Use **demo/images/** as Jetson input. The CSV and YOLO labels use crop coordinates derived from your original-photo boxes.

In [ ]:
CLASS_ORDER = []  # Replace with the six CLASSES in the actual training ID order.
from PIL import ImageDraw

if CLASS_ORDER and (len(CLASS_ORDER)!=len(CLASSES) or set(CLASS_ORDER)!=set(CLASSES)):
    raise ValueError("Use all six canonical class spellings once, in actual training ID order.")
if set(REVIEW)-crops.keys():
    raise ValueError(f"Review includes unknown IDs: {set(REVIEW)-crops.keys()}")
reviews = {key:checked(key) for key in crops}
with TemporaryDirectory() as temporary:
    root = Path(temporary)
    for name in ('demo/images','evaluation/images','evaluation/labels','annotated/images'):
        (root/name).mkdir(parents=True)
    manifest, annotations = [], []
    for key,image in crops.items():
        photo,x,y,side,group = CROPS[key]
        status,objects = reviews[key]
        name = key+'.png'
        image.save(root/'demo/images'/name)
        manifest.append([key,source_names[photo],group,x,y,side,TARGET,USE_AUTOCONTRAST,status])
        for label,*box in objects:
            annotations.append([key,label,*box])
        if status == 'positive':
            overlay = image.convert('RGB')
            drawing = ImageDraw.Draw(overlay)
            for label,x1,y1,x2,y2 in objects:
                rectangle = (math.floor(x1), math.floor(y1), math.ceil(x2)-1, math.ceil(y2)-1)
                drawing.rectangle(rectangle, outline='lime', width=1)
            overlay.save(root/'annotated/images'/name)
        if status in {'positive','negative'}:
            shutil.copyfile(root/'demo/images'/name,root/'evaluation/images'/name)
            if CLASS_ORDER:
                lines = []
                for label,x1,y1,x2,y2 in objects:
                    values = ((x1+x2)/(2*TARGET),(y1+y2)/(2*TARGET),
                              (x2-x1)/TARGET,(y2-y1)/TARGET)
                    lines.append(f"{CLASS_ORDER.index(label)} "+" ".join(f"{v:.6f}" for v in values))
                (root/'evaluation/labels'/f'{key}.txt').write_text(''.join(line+'\n' for line in lines))
    write_csv(root/'manifest.csv',
              ['id','source','source_group','left','top','side','output_size','autocontrast','status'],manifest)
    write_csv(root/'boxes.csv',['id','class','left','top','right','bottom'],annotations)
    (root/'review.json').write_text(json.dumps(REVIEW, indent=2))
    (root/'class_order.json').write_text(json.dumps(CLASS_ORDER, indent=2))
    archive = shutil.make_archive(str(DATA_DIR / 'phone_steel_external'),'zip',root)
print('Reviewed evaluation images:', sum(status in {'positive','negative'} for status,_ in reviews.values()))
print('Images saved with visible boxes:', sum(status=='positive' for status,_ in reviews.values()))
print('YOLO label export enabled:', bool(CLASS_ORDER))
print("Saved to Drive:", archive)

## Public external benchmark alongside your phone set
Use GC10-DET as a separate batch. Start with its **inclusion** class, the clearest shared label with NEU-DET. Do not label drilled holes as inclusions or merge rolled pit with pitted surface without a grounded mapping. Keep the GC10 image IDs and source annotations. Select a batch before inference, freeze thresholds and preprocessing, and report which classes are scored.

- For a quick clip, select a few readable steel regions, convert to grayscale and crop them to square before resizing. This notebook's crop dictionary uses fixed pixels for **your phone photos**. To add other photos, assign unique IDs in `photos` and `source_names`, define square regions in `CROPS`, then rerun from cropping. Keep exports for phone and GC10 separate.
- For evaluation, prefer a fixed field of view or fixed tiles and transfer the existing annotation boxes. For a square crop at `(left, top)` with side `S`, subtract the crop origin, clip to `[0,S]`, and multiply by `200/S`. Set an explicit policy for boxes partially outside the crop, and review converted labels before scoring. This notebook exports manual annotations entered on the original phone photos and projected onto the final crops.
- Keep images of the same physical piece together. Report the phone-photo results and GC10 transfer results independently. For the Jetson demo, start with one known-good NEU-DET image, then run the external sample with the same detector and confidence threshold.